## Logistic regression

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define results DataFrame structure
results_columns = ['Model', 'Accuracy', 'Recall', 'Specificity',
                  'Precision', 'F1', 'Time']
df_results = pd.DataFrame(columns=results_columns)

# Configuration parameters
VIF_THRESHOLD = 10
PVAL_THRESHOLD = 0.01
MAX_ITERATIONS = 100

def perform_feature_selection(X_train, y_train):
    """
    Perform iterative feature selection using p-values and VIF analysis.

    Args:
        X_train: Training features DataFrame
        y_train: Training labels Series

    Returns:
        Tuple: (Final model, List of selected features)
    """
    current_features = X_train.columns.tolist()
    removed_features = []

    # First phase: Remove features with high p-values
    while True:
        model = sm.Logit(y_train, X_train[current_features]).fit(
            disp=0,
            maxiter=MAX_ITERATIONS
        )

        # Get highest p-value feature
        p_values = model.pvalues
        max_p_feature = p_values.idxmax()
        max_p_value = p_values.max()

        if max_p_value < PVAL_THRESHOLD:
            break

        print(f"Removing feature: {max_p_feature} (p-value: {max_p_value:.3f})")
        current_features.remove(max_p_feature)

    # Second phase: Remove features with high multicollinearity
    while True:
        # Calculate VIF for remaining features
        vif_data = pd.DataFrame()
        vif_data['Feature'] = current_features
        vif_data['VIF'] = [variance_inflation_factor(X_train[current_features].values, i)
                          for i in range(len(current_features))]

        max_vif = vif_data['VIF'].max()
        max_vif_feature = vif_data.loc[vif_data['VIF'].idxmax(), 'Feature']

        if max_vif < VIF_THRESHOLD:
            print("All VIF values below threshold")
            break

        print(f"Removing feature: {max_vif_feature} (VIF: {max_vif:.1f})")
        current_features.remove(max_vif_feature)

    return model, current_features

# Main cross-validation loop
for fold_idx in tqdm(range(5), desc="Processing folds"):
    # Split data into train/test sets
    test_data = pd.concat([D[fold_idx]])
    train_data = pd.concat([d for i, d in enumerate(D) if i != fold_idx])

    # Prepare datasets
    y_test = test_data['label_binary']
    X_test = test_data.drop(columns=['label_binary', 'n_image', 'label_multi'])

    y_train = train_data['label_binary']
    X_train = train_data.drop(columns=['label_binary', 'n_image', 'label_multi'])

    # Encode labels to numerical values
    label_mapping = {'no corrosion': 0, 'corrosion': 1}
    y_train = y_train.map(label_mapping)
    y_test = y_test.map(label_mapping)

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Convert back to DataFrames with original column names
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

    # Feature selection process
    final_model, selected_features = perform_feature_selection(X_train_scaled, y_train)

    # Make predictions
    start_time = time.time()
    X_test_final = X_test_scaled[selected_features]
    y_pred_proba = final_model.predict(X_test_final)
    y_pred = (y_pred_proba >= 0.5).astype(int)
    elapsed_time = time.time() - start_time

    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
    print(f"\nConfusion matrix for fold {fold_idx+1}:\n{cm}")

    fold_metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred, pos_label=1),
        'specificity': recall_score(y_test, y_pred, pos_label=0),
        'precision': precision_score(y_test, y_pred, pos_label=1),
        'f1': f1_score(y_test, y_pred, pos_label=1),
        'time': elapsed_time
    }

    # Store metrics
    df_results = pd.concat([
        df_results,
        pd.DataFrame([{
            'Model': 'Logistic Regression',
            'Accuracy': fold_metrics['accuracy'],
            'Recall': fold_metrics['recall'],
            'Specificity': fold_metrics['specificity'],
            'Precision': fold_metrics['precision'],
            'F1': fold_metrics['f1'],
            'Time': fold_metrics['time']
        }])
    ], ignore_index=True)

# Calculate mean metrics across all folds
mean_results = {
    'Model': 'Logistic Regression',
    'Accuracy': df_results['Accuracy'].mean(),
    'Recall': df_results['Recall'].mean(),
    'Specificity': df_results['Specificity'].mean(),
    'Precision': df_results['Precision'].mean(),
    'F1': df_results['F1'].mean(),
    'Time': df_results['Time'].mean()
}

# Add mean results to DataFrame
df_results = pd.concat([
    df_results,
    pd.DataFrame([mean_results])
], ignore_index=True)

print("\nFinal Results:")
print(df_results.round(3))